In [1]:
import requests
import pandas as pd

url = "https://api.coingecko.com/api/v3/coins/markets"
params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1
}

response = requests.get(url, params=params)
data = response.json()
df = pd.DataFrame(data)

# Step 3: select only the columns we care about
columns_to_keep = [
    "id", "symbol", "name", "current_price",
    "market_cap", "total_volume", "price_change_percentage_24h"
]
df_clean = df[columns_to_keep]

# rename for clarity
df_clean = df_clean.rename(columns={
    "current_price": "price_usd",
    "price_change_percentage_24h": "change_24h_pct"
})

# check for missing values
print(df_clean.isnull().sum())

# drop any rows with missing critical data (e.g. no price)
df_clean = df_clean.dropna(subset=["price_usd"])

print(df_clean)

id                0
symbol            0
name              0
price_usd         0
market_cap        0
total_volume      0
change_24h_pct    0
dtype: int64
             id      symbol          name     price_usd     market_cap  \
0       bitcoin         btc       Bitcoin  84650.000000  1700625092054   
1      ethereum         eth      Ethereum   2708.880000   330707452338   
2        tether        usdt        Tether      0.999800   183785296424   
3   binancecoin         bnb           BNB    777.080000   103477815952   
4        ripple         xrp           XRP      1.530000    96167420303   
5      usd-coin        usdc          USDC      0.999868    75258887649   
6        solana         sol        Solana    123.770000    72752990618   
7          tron         trx          TRON      0.333885    31708090084   
8         zcash         zec         Zcash   1651.240000    28020145332   
9  figure-heloc  figr_heloc  Figure Heloc      1.062000    24610425446   

   total_volume  change_24h_pct 

In [2]:
columns_to_keep = [
    "id", "symbol", "name", "current_price",
    "market_cap", "total_volume", "price_change_percentage_24h"
]
df_clean = df[columns_to_keep]

df_clean = df_clean.rename(columns={
    "current_price": "price_usd",
    "price_change_percentage_24h": "change_24h_pct"
})

print(df_clean.isnull().sum())

df_clean = df_clean.dropna(subset=["price_usd"])
print(df_clean)

id                0
symbol            0
name              0
price_usd         0
market_cap        0
total_volume      0
change_24h_pct    0
dtype: int64
             id      symbol          name     price_usd     market_cap  \
0       bitcoin         btc       Bitcoin  84650.000000  1700625092054   
1      ethereum         eth      Ethereum   2708.880000   330707452338   
2        tether        usdt        Tether      0.999800   183785296424   
3   binancecoin         bnb           BNB    777.080000   103477815952   
4        ripple         xrp           XRP      1.530000    96167420303   
5      usd-coin        usdc          USDC      0.999868    75258887649   
6        solana         sol        Solana    123.770000    72752990618   
7          tron         trx          TRON      0.333885    31708090084   
8         zcash         zec         Zcash   1651.240000    28020145332   
9  figure-heloc  figr_heloc  Figure Heloc      1.062000    24610425446   

   total_volume  change_24h_pct 

In [3]:
import pyodbc
print(pyodbc.drivers())

['SQL Server', 'ODBC Driver 17 for SQL Server', 'ODBC Driver 18 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)']


In [4]:
from sqlalchemy import create_engine

server = "localhost\\SQLEXPRESS"       # or your SQL Server instance name, e.g. "localhost\\SQLEXPRESS"
database = "crypto_pipeline" # we'll create this database in a sec
driver = "ODBC+Driver+17+for+SQL+Server"

connection_string = f"mssql+pyodbc://@{server}/{database}?trusted_connection=yes&driver={driver}"
engine = create_engine(connection_string)

In [5]:
with engine.connect() as conn:
    print("Connected successfully!")

Connected successfully!


C:\Users\prate_onq7uva\AppData\Local\Temp\ipykernel_15208\177040175.py:1: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


In [6]:
df_clean.to_sql("coin_prices", con=engine, if_exists="replace", index=False)
print("Data written to SQL Server!")

Data written to SQL Server!


In [7]:
query = """
SELECT name, symbol, price_usd, change_24h_pct
FROM coin_prices
ORDER BY change_24h_pct DESC
"""

df_result = pd.read_sql(query, con=engine)
print(df_result)

           name      symbol     price_usd  change_24h_pct
0         Zcash         zec   1651.240000         8.44392
1  Figure Heloc  figr_heloc      1.062000         3.68795
2        Solana         sol    123.770000         3.10061
3      Ethereum         eth   2708.880000         0.97240
4       Bitcoin         btc  84650.000000         0.75089
5           BNB         bnb    777.080000         0.74999
6        Tether        usdt      0.999800        -0.00514
7          USDC        usdc      0.999868        -0.00523
8           XRP         xrp      1.530000        -0.72875
9          TRON         trx      0.333885        -0.87589


In [8]:
# average price change across all coins
query2 = "SELECT AVG(change_24h_pct) AS avg_change FROM coin_prices"
print(pd.read_sql(query2, con=engine))

# top 3 by market cap
query3 = "SELECT TOP 3 name, market_cap FROM coin_prices ORDER BY market_cap DESC"
print(pd.read_sql(query3, con=engine))

   avg_change
0    1.609075
       name     market_cap
0   Bitcoin  1700625092054
1  Ethereum   330707452338
2    Tether   183785296424


In [9]:
df_result.to_excel("crypto_report.xlsx", index=False, sheet_name="Report")
print("Excel report created!")

Excel report created!


In [10]:
from openpyxl.styles import Font, PatternFill
from openpyxl.utils import get_column_letter

with pd.ExcelWriter("crypto_report.xlsx", engine="openpyxl") as writer:
    df_result.to_excel(writer, index=False, sheet_name="Report")

    ws = writer.sheets["Report"]

    # bold + colored header row
    header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
    header_font = Font(bold=True, color="FFFFFF")
    for col_num, col_name in enumerate(df_result.columns, start=1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill

    # auto-fit column widths
    for col_num, col_name in enumerate(df_result.columns, start=1):
        max_len = max(df_result[col_name].astype(str).map(len).max(), len(col_name))
        ws.column_dimensions[get_column_letter(col_num)].width = max_len + 2

print("Formatted Excel report created!")

Formatted Excel report created!


In [11]:
import logging

logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

In [12]:
try:
    response = requests.get(url, params=params)
    response.raise_for_status()   # raises an exception if status_code is 4xx/5xx
    data = response.json()
    logging.info(f"Fetched {len(data)} records from API")
except requests.exceptions.RequestException as e:
    logging.error(f"API call failed: {e}")
    raise   # stop the script here, since nothing else can run without data

In [13]:
try:
    df = pd.DataFrame(data)
    df_clean = df[columns_to_keep].rename(columns={
        "current_price": "price_usd",
        "price_change_percentage_24h": "change_24h_pct"
    })
    df_clean = df_clean.dropna(subset=["price_usd"])
    logging.info(f"Cleaned data: {df_clean.shape[0]} rows, {df_clean.shape[1]} columns")
except Exception as e:
    logging.error(f"Data cleaning failed: {e}")
    raise

In [14]:
try:
    df_clean.to_sql("coin_prices", con=engine, if_exists="replace", index=False)
    logging.info("Data written to SQL Server successfully")
except Exception as e:
    logging.error(f"Database write failed: {e}")
    raise

In [15]:
try:
    with pd.ExcelWriter("crypto_report.xlsx", engine="openpyxl") as writer:
        df_result.to_excel(writer, index=False, sheet_name="Report")
        ws = writer.sheets["Report"]
        # (formatting code from before goes here)
    logging.info("Excel report generated successfully")
except Exception as e:
    logging.error(f"Excel report generation failed: {e}")
    raise